# Diagnóstico y mejora del modelo delay_30m — Distribution shift feb 2026

El modelo LightGBM obtenía **MAE≈130s** en validación (oct-dic 2025) pero **MAE≈163s** en test (feb 2026).  
Este notebook analiza las causas del empeoramiento y evalúa estrategias de mitigación.

**Estructura:**
1. Carga de datos
2. Análisis del distribution shift
3. Análisis de alertas MTA
4. Impacto de la nevada (23-24 feb)
5. MAE real del modelo — versión completa (ene-dic 2025 + ene 2026)
6. Solución: ventana deslizante (jul-dic 2025 + ene 2026)
7. Importancia de features vs drift
8. Conclusiones

## Glosario de variables y abreviaturas

| Abreviatura | Nombre completo | Fórmula | Qué mide |
|---|---|---|---|
| **RAM** | Retraso Absoluto Medio | `mean(|target|)` | Magnitud media de los retrasos reales en los datos. Describe el fenómeno, **no** el error del modelo. |
| **MAE** | Mean Absolute Error | `mean(|predicción − real|)` | Error medio de predicción del modelo. Es la métrica principal de evaluación. |
| **RMSE** | Root Mean Squared Error | `sqrt(mean((predicción − real)²))` | Error medio penalizando más los errores grandes. |
| **R²** | Coeficiente de determinación | `1 − SS_res / SS_tot` | Proporción de varianza del target explicada por el modelo (0 = nada, 1 = perfecto). |
| **Sesgo** | Sesgo sistemático | `mean(predicción − real)` | Si es positivo, el modelo sobreestima; si es negativo, subestima. |
| **drift** | Desplazamiento de distribución | `|media_test − media_train| / std_train` | Cuántas desviaciones típicas se ha desplazado una feature entre train y test (en sigmas). |
| **df_val** | Datos de validación | — | Oct–dic 2025. Periodo usado para seleccionar hiperparámetros. |
| **df_ene** | Enero 2026 | — | Siempre incluido en train. |
| **df_test** | Datos de test | — | Febrero 2026. Datos nuevos nunca vistos por el modelo. |
| **df_mar** | Marzo 2026 | — | Solo para diagnóstico del fallo del pipeline de alertas MTA. No entra en train ni test. |

In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error, r2_score

from src.common.minio_client import download_df_parquet

ACCESS_KEY = os.environ['MINIO_ACCESS_KEY']
SECRET_KEY = os.environ['MINIO_SECRET_KEY']

DATA_TEMPLATE = 'grupo5/final/year={year}/month={month:02d}/dataset_final.parquet'
TARGET        = 'target_delay_30m'
FILTER_COL    = 'scheduled_time_to_end'
FILTER_VAL    = 1800

CAT_FEATURES  = ['route_id', 'direction', 'category', 'tipo_referente']
STOP_ID_COL   = 'stop_id'

print('Imports OK.')

## 1. Carga de datos

In [ ]:
def load_months(year, months, label):
    dfs = []
    for month in months:
        path = DATA_TEMPLATE.format(year=year, month=month)
        try:
            df = download_df_parquet(ACCESS_KEY, SECRET_KEY, path)
            df = df[df['is_unscheduled'] == False]
            df = df.dropna(subset=[TARGET])
            df = df[df[FILTER_COL] >= FILTER_VAL]
            df['_periodo'] = label
            print(f'  ✓ {year}-{month:02d}: {len(df):,} filas')
            dfs.append(df)
        except Exception as e:
            print(f'  ✗ {year}-{month:02d}: {e}')
    return pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()

print('Cargando validación (oct-dic 2025)...')
df_val = load_months(2025, range(10, 13), 'val (oct-dic 2025)')

print('\nCargando ene 2026...')
df_ene = load_months(2026, range(1, 2), 'ene 2026')

print('\nCargando test (feb 2026)...')
df_test = load_months(2026, range(2, 3), 'test (feb 2026)')

# Marzo 2026: SOLO para diagnóstico del fallo del pipeline de alertas MTA.
# No se usa en entrenamiento ni en evaluación del modelo.
print('\nCargando mar 2026 (solo diagnóstico alertas)...')
df_mar = load_months(2026, range(3, 4), 'mar 2026 [diagnóstico]')

print(f'\nResumen:')
print(f'  val (oct-dic 2025): {len(df_val):,} filas')
print(f'  ene 2026:           {len(df_ene):,} filas')
print(f'  test (feb 2026):    {len(df_test):,} filas')
print(f'  mar 2026 (diag):    {len(df_mar):,} filas  ← solo alertas, no entra en train/test')

## 2. Análisis del distribution shift

> Nota : en esta sección se analiza el retraso absoluto medio (`mean(|target|)`),
> que mide la magnitud típica de los retrasos en cada periodo. **No es el MAE del modelo** —
> el MAE real (error de predicción) se calcula en la Sección 5.

In [ ]:
# 2.1 Estadísticas del target por periodo
print('Estadísticas del target (target_delay_30m):')
stats = pd.DataFrame({
    'val (oct-dic 2025)': df_val[TARGET].describe(),
    'ene 2026':           df_ene[TARGET].describe(),
    'test (feb 2026)':    df_test[TARGET].describe(),
})
print(stats.round(2).to_string())

print('\nRetraso absoluto medio por periodo (≠ MAE del modelo):')
for label, df in [('val (oct-dic 2025)', df_val), ('ene 2026', df_ene), ('test (feb 2026)', df_test)]:
    ram = df[TARGET].abs().mean()
    print(f'  {label}: {ram:.1f}s')

In [ ]:
# 2.2 Distribución del target por periodo
datasets = [
    ('val (oct-dic 2025)', df_val, 'steelblue'),
    ('ene 2026',           df_ene, 'seagreen'),
    ('test (feb 2026)',    df_test, 'tomato'),
]

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, (label, df, color) in zip(axes, datasets):
    data = df[TARGET].clip(-600, 1800)
    ax.hist(data, bins=80, color=color, alpha=0.8, edgecolor='none')
    ax.axvline(data.median(), color='black', linewidth=1.2, linestyle='--',
               label=f'Mediana: {data.median():.0f}s')
    ax.set_title(label)
    ax.set_xlabel('target_delay_30m (s)')
    ax.legend(fontsize=8)
plt.suptitle('Distribución del target por periodo', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# 2.3 Empeoramiento por línea: retraso absoluto medio val → test
# Nota: se compara la magnitud media de retrasos, no el error del modelo
ram_val  = df_val.groupby('route_id')[TARGET].apply(lambda x: x.abs().mean()).rename('ram_val')
ram_test = df_test.groupby('route_id')[TARGET].apply(lambda x: x.abs().mean()).rename('ram_test')
comp = pd.concat([ram_val, ram_test], axis=1).dropna()
comp['delta'] = comp['ram_test'] - comp['ram_val']
comp = comp.sort_values('delta', ascending=False)

print('Cambio en retraso absoluto medio por línea (val oct-dic 2025 → test feb 2026):')
print('(Positivo = más retraso en feb 2026 que en validación)\n')
print(comp.round(1).to_string())

In [ ]:
# 2.4 Shift en las features de retraso más importantes
FEATS_SHIFT = ['delay_seconds', 'lagged_delay_1', 'route_rolling_delay']

print('Shift en features clave (train jul-dic 2025 vs test feb 2026):')
print(f'{"Feature":<25} {"Media val":>12} {"Media test":>12} {"Δ %":>8}')
print('-' * 60)
for feat in FEATS_SHIFT:
    if feat in df_val.columns and feat in df_test.columns:
        m_val  = df_val[feat].mean()
        m_test = df_test[feat].mean()
        delta_pct = (m_test - m_val) / abs(m_val) * 100
        print(f'{feat:<25} {m_val:>12.1f} {m_test:>12.1f} {delta_pct:>+8.1f}%')

## 3. Análisis de alertas MTA

Se verifica que las columnas de alertas no estén rotas en febrero 2026  
(en marzo 2026 estaban al 100% nulas por un fallo del pipeline `data.ny.gov`).

In [ ]:
# 3.1 % de nulos por columna de alerta (incluyendo mar-2026 para mostrar el fallo)
ALERT_COLS = ['alert_in_next_15m', 'alert_in_next_30m',
              'seconds_to_next_alert', 'seconds_since_last_alert']
alert_cols_ok = [c for c in ALERT_COLS if c in df_val.columns]

print('=== % de nulos por columna de alerta ===')
periodos = [
    ('val (oct-dic 2025)', df_val),
    ('ene 2026',           df_ene),
    ('test (feb 2026)',    df_test),
    ('mar 2026 [ROTO]',   df_mar),
]
for col in alert_cols_ok:
    print(f'\n{col}:')
    for label, df in periodos:
        if df.empty or col not in df.columns:
            continue
        pct = df[col].isnull().mean() * 100
        flag = ' ⚠ PIPELINE ROTO' if pct == 100 else ''
        print(f'  {label}: {pct:.1f}% nulos{flag}')

In [ ]:
# 3.2 Nulos de seconds_to_next_alert mes a mes (confirma rotura en marzo 2026)
print('=== seconds_to_next_alert: % nulos por mes ===')
todos = pd.concat([df_val, df_ene, df_test, df_mar], ignore_index=True)
todos['_year']  = pd.to_datetime(todos['date']).dt.year
todos['_month'] = pd.to_datetime(todos['date']).dt.month

for (year, month), grp in todos.groupby(['_year', '_month']):
    pct = grp['seconds_to_next_alert'].isnull().mean() * 100
    flag = ' ⚠ PIPELINE ROTO' if pct == 100 else ''
    print(f'  {year}-{month:02d}: {pct:.1f}% nulos  ({len(grp):,} filas){flag}')

In [ ]:
# 3.3 Correlación alertas vs target (señal real vs ruido)
print('=== Correlación de alertas con el target ===')
for label, df in [('val (oct-dic 2025)', df_val), ('test (feb 2026)', df_test)]:
    print(f'\n{label}:')
    for col in alert_cols_ok:
        corr = df[[col, TARGET]].dropna().corr().loc[col, TARGET]
        print(f'  {col}: {corr:.4f}')

## 4. Impacto de la nevada (23-24 feb 2026)

Se analizó si los días de nevada inflaban artificialmente el retraso absoluto medio del periodo de test.

In [ ]:
NEVADA_DATES = ['2026-02-23', '2026-02-24']
df_test['_date'] = pd.to_datetime(df_test['date'])

df_nevada    = df_test[df_test['_date'].dt.strftime('%Y-%m-%d').isin(NEVADA_DATES)]
df_sin_nieve = df_test[~df_test['_date'].dt.strftime('%Y-%m-%d').isin(NEVADA_DATES)]

ram_global    = df_test[TARGET].abs().mean()
ram_nevada    = df_nevada[TARGET].abs().mean()
ram_sin_nieve = df_sin_nieve[TARGET].abs().mean()
ram_val       = df_val[TARGET].abs().mean()

print(f'Filas en días de nevada: {len(df_nevada):,} ({len(df_nevada)/len(df_test)*100:.1f}% del test)\n')
print('Retraso absoluto medio (≠ MAE del modelo):')
print(f'  Referencia val (oct-dic 2025):  {ram_val:.1f}s')
print(f'  Test global (feb 2026):         {ram_global:.1f}s')
print(f'  Solo días de nevada:            {ram_nevada:.1f}s')
print(f'  Sin días de nevada:             {ram_sin_nieve:.1f}s')
print(f'\n→ La nevada explica solo {ram_global - ram_sin_nieve:.1f}s del incremento total de {ram_global - ram_val:.1f}s')
print('→ El grueso del empeoramiento es distribution shift, no la nevada.')

## 5. MAE real del modelo — versión completa (ene-dic 2025 + ene 2026)

A partir de aquí se calculan predicciones reales del modelo para obtener el **MAE verdadero**  
(error de predicción), distinguiéndolo del retraso absoluto medio analizado anteriormente.

In [ ]:
EXCLUDE_M = {
    'date', 'match_key', 'stop_id', 'merge_time', 'timestamp_start',
    'service_date', 'trip_uid', 'is_unscheduled',
    'target_delay_10m', 'target_delay_20m', 'target_delay_30m',
    'target_delay_45m', 'target_delay_60m', 'target_delay_end',
    'delta_delay_10m', 'delta_delay_20m', 'delta_delay_30m',
    'delta_delay_45m', 'delta_delay_60m', 'delta_delay_end',
    'alert_in_next_15m', 'alert_in_next_30m', 'seconds_to_next_alert',
    'delay_minutes', 'scheduled_time', 'actual_time',
    '_periodo', '_date', '_route_id',
}

LGBM_PARAMS = {
    'objective': 'regression_l1', 'metric': 'mae',
    'learning_rate': 0.05, 'num_leaves': 511, 'max_depth': 16,
    'min_child_samples': 100, 'min_split_gain': 0.37042771510661165,
    'feature_fraction': 0.7426288737567357, 'bagging_fraction': 0.8165370010747616,
    'bagging_freq': 5, 'reg_alpha': 1.5346393797283635,
    'reg_lambda': 1.2926631392622208, 'n_jobs': -1, 'verbose': -1, 'seed': 42,
}
NUM_BOOST_ROUND = 4260

def encode_cat(tr, te):
    for col in CAT_FEATURES:
        if col not in tr.columns: continue
        vocab = {v: i for i, v in enumerate(tr[col].astype(str).unique())}
        tr[col] = tr[col].astype(str).map(vocab).astype(int)
        te[col] = te[col].astype(str).map(vocab).fillna(-1).astype(int)
    return tr, te

def add_te(tr, te, col, target):
    means = tr.groupby(col)[target].mean()
    tr[f'{col}_target_enc'] = tr[col].map(means)
    te[f'{col}_target_enc'] = te[col].map(means).fillna(tr[target].mean())
    return tr, te

def add_feats(df):
    if 'lagged_delay_1' in df.columns and 'delay_seconds' in df.columns:
        df['delay_velocity'] = df['delay_seconds'] - df['lagged_delay_1']
    if 'lagged_delay_1' in df.columns and 'lagged_delay_2' in df.columns:
        df['delay_acceleration'] = (
            (df['delay_seconds'] - df['lagged_delay_1'])
            - (df['lagged_delay_1'] - df['lagged_delay_2'])
        )
    if 'delay_seconds' in df.columns and 'stops_to_end' in df.columns:
        df['delay_x_stops_remaining'] = df['delay_seconds'] * df['stops_to_end']
    if 'delay_seconds' in df.columns and 'scheduled_time_to_end' in df.columns:
        df['delay_ratio'] = df['delay_seconds'] / (df['scheduled_time_to_end'] + 1)
    return df

def get_feats(df):
    return [c for c in df.columns if c not in EXCLUDE_M and c != TARGET]

def load_train(year, months):
    dfs = []
    for month in months:
        path = DATA_TEMPLATE.format(year=year, month=month)
        try:
            df = download_df_parquet(ACCESS_KEY, SECRET_KEY, path)
            df = df[df['is_unscheduled'] == False]
            df = df.dropna(subset=[TARGET])
            df = df[df[FILTER_COL] >= FILTER_VAL]
            for col in CAT_FEATURES:
                if col in df.columns:
                    df[col] = df[col].astype('category')
            print(f'  {year}-{month:02d}: {len(df):,} filas')
            dfs.append(df)
        except Exception as e:
            print(f'  {year}-{month:02d}: error ({e})')
    return pd.concat(dfs, ignore_index=True)

print('Funciones listas.')

In [ ]:
# Cargar train: ene-dic 2025 + ene 2026 (versión completa)
print('Cargando train ene-dic 2025...')
dft25_full = load_train(2025, range(1, 13))
print('Cargando train ene 2026...')
dft26      = load_train(2026, range(1, 2))
df_tr_full = pd.concat([dft25_full, dft26], ignore_index=True)
print(f'Total train completo: {len(df_tr_full):,} filas')

In [ ]:
df_tr_f = df_tr_full.copy()
df_te_f = df_test.copy()
df_te_f['_route_id'] = df_te_f['route_id'].astype(str)

df_tr_f, df_te_f = encode_cat(df_tr_f, df_te_f)
df_tr_f, df_te_f = add_te(df_tr_f, df_te_f, STOP_ID_COL, TARGET)
df_tr_f = add_feats(df_tr_f)
df_te_f = add_feats(df_te_f)

feats_f = get_feats(df_tr_f)
print(f'Entrenando modelo completo ({NUM_BOOST_ROUND} iters, {len(feats_f)} features)...')
model_full = lgb.train(
    LGBM_PARAMS,
    lgb.Dataset(df_tr_f[feats_f], label=df_tr_f[TARGET]),
    num_boost_round=NUM_BOOST_ROUND,
    callbacks=[lgb.log_evaluation(1000)],
)

df_tr_f['_pred'] = model_full.predict(df_tr_f[feats_f])
df_te_f['_pred'] = model_full.predict(df_te_f[feats_f])

mae_full_train = mean_absolute_error(df_tr_f[TARGET], df_tr_f['_pred'])
r2_full_train  = r2_score(df_tr_f[TARGET], df_tr_f['_pred'])
mae_full       = mean_absolute_error(df_te_f[TARGET], df_te_f['_pred'])
r2_full        = r2_score(df_te_f[TARGET], df_te_f['_pred'])

print(f'\nMétricas modelo completo:')
print(f'  Train (ene-dic 2025 + ene 2026): MAE={mae_full_train:.2f}s  |  R²={r2_full_train:.4f}')
print(f'  Test  (feb 2026):                MAE={mae_full:.2f}s  |  R²={r2_full:.4f}')
print(f'\n→ Gap train/test de {mae_full - mae_full_train:.1f}s indica overfitting al periodo histórico.')

In [ ]:
# Sobreestimación sistemática por línea
print('=== MAE y sesgo por línea — modelo completo ===\n')
by_line_full = df_te_f.groupby('_route_id').apply(lambda g: pd.Series({
    'n_filas':  len(g),
    'mae_s':    mean_absolute_error(g[TARGET], g['_pred']),
    'r2':       r2_score(g[TARGET], g['_pred']),
    'sesgo_s':  (g['_pred'] - g[TARGET]).mean(),
})).sort_values('mae_s', ascending=False)
print(by_line_full.round(1).to_string())
print(f'\n→ Sesgo positivo = sobreestima. El modelo predice más retraso del que ocurre en feb 2026.')

## 6. Solución: ventana deslizante (jul-dic 2025 + ene 2026)

### Motivación

El análisis de las secciones anteriores revela que el empeoramiento del modelo no se debe a  
un fallo puntual sino a **concept drift**: el régimen de retrasos del metro en 2026 es  
sistemáticamente más alto que en 2025 (+15% en `delay_seconds`, +17% en `route_rolling_delay`).

El modelo entrenado con todo 2025 aprendió que valores de retraso actuales de ~100s eran  
"casos altos" y predecía futuros aún más altos — produciendo sobreestimaciones de +70s (línea B)  
y +46s (línea E).

### Solución: ventana deslizante

En lugar de usar todos los datos históricos, se entrena solo con los **últimos 7 meses**  
(jul-dic 2025 + ene 2026). Esto tiene dos efectos:
- El train está temporalmente más cerca del test → menor drift de distribución
- Los patrones del 2º semestre de 2025 (donde los retrasos ya eran más altos) pesan más

Es la estrategia estándar en ML de producción para mitigar concept drift sin necesitar  
datos del periodo de test.

In [ ]:
# Cargar train: jul-dic 2025 + ene 2026 (ventana deslizante)
print('Cargando train jul-dic 2025 (sliding window)...')
dft25_sw = load_train(2025, range(7, 13))
print('Cargando train ene 2026...')
df_tr_sw = pd.concat([dft25_sw, dft26], ignore_index=True)
print(f'Total train sliding: {len(df_tr_sw):,} filas')
print(f'(vs {len(df_tr_full):,} del modelo completo — {len(df_tr_sw)/len(df_tr_full)*100:.0f}% de los datos)')

In [ ]:
df_tr_s = df_tr_sw.copy()
df_te_s = df_test.copy()
df_te_s['_route_id'] = df_te_s['route_id'].astype(str)
df_te_s['_date']     = pd.to_datetime(df_te_s['date'])

df_tr_s, df_te_s = encode_cat(df_tr_s, df_te_s)
df_tr_s, df_te_s = add_te(df_tr_s, df_te_s, STOP_ID_COL, TARGET)
df_tr_s = add_feats(df_tr_s)
df_te_s = add_feats(df_te_s)

feats_s = get_feats(df_tr_s)
print(f'Entrenando modelo sliding ({NUM_BOOST_ROUND} iters, {len(feats_s)} features)...')
model_sw = lgb.train(
    LGBM_PARAMS,
    lgb.Dataset(df_tr_s[feats_s], label=df_tr_s[TARGET]),
    num_boost_round=NUM_BOOST_ROUND,
    callbacks=[lgb.log_evaluation(1000)],
)

df_tr_s['_pred'] = model_sw.predict(df_tr_s[feats_s])
df_te_s['_pred'] = model_sw.predict(df_te_s[feats_s])

mae_sw_train = mean_absolute_error(df_tr_s[TARGET], df_tr_s['_pred'])
r2_sw_train  = r2_score(df_tr_s[TARGET], df_tr_s['_pred'])
mae_sw       = mean_absolute_error(df_te_s[TARGET], df_te_s['_pred'])
r2_sw        = r2_score(df_te_s[TARGET], df_te_s['_pred'])

print(f'\nMétricas modelo sliding:')
print(f'  Train (jul-dic 2025 + ene 2026): MAE={mae_sw_train:.2f}s  |  R²={r2_sw_train:.4f}')
print(f'  Test  (feb 2026):                MAE={mae_sw:.2f}s  |  R²={r2_sw:.4f}')
print(f'\nMejora vs modelo completo (test): {mae_full - mae_sw:.1f}s ({(mae_full - mae_sw)/mae_full*100:.1f}%)')
print(f'Gap train/test sliding:           {mae_sw - mae_sw_train:.1f}s  (vs {mae_full - mae_full_train:.1f}s del completo)')

In [ ]:
# MAE y sesgo por línea — modelo sliding
print('=== MAE y sesgo por línea — modelo sliding ===\n')
by_line_sw = df_te_s.groupby('_route_id').apply(lambda g: pd.Series({
    'n_filas':  len(g),
    'mae_s':    mean_absolute_error(g[TARGET], g['_pred']),
    'r2':       r2_score(g[TARGET], g['_pred']),
    'sesgo_s':  (g['_pred'] - g[TARGET]).mean(),
})).sort_values('mae_s', ascending=False)
print(by_line_sw.round(1).to_string())

In [ ]:
# Comparación sesgo por línea: completo vs sliding
print('=== Eliminación del sesgo sistemático: completo → sliding ===\n')
lineas_foco = ['B', 'E', 'F', 'M']
print(f'{"Línea":<8} {"MAE completo":>14} {"MAE sliding":>13} {"Sesgo completo":>16} {"Sesgo sliding":>15}')
print('-' * 70)
for linea in lineas_foco:
    g_f = df_te_f[df_te_f['_route_id'] == linea]
    g_s = df_te_s[df_te_s['_route_id'] == linea]
    if len(g_f) == 0 or len(g_s) == 0:
        continue
    mae_f  = mean_absolute_error(g_f[TARGET], g_f['_pred'])
    mae_s  = mean_absolute_error(g_s[TARGET], g_s['_pred'])
    ses_f  = (g_f['_pred'] - g_f[TARGET]).mean()
    ses_s  = (g_s['_pred'] - g_s[TARGET]).mean()
    print(f'{linea:<8} {mae_f:>14.1f}s {mae_s:>13.1f}s {ses_f:>+16.1f}s {ses_s:>+15.1f}s')

In [ ]:
# Impacto de exclusiones sobre el MAE real (modelo sliding)
LINEAS_AF = ['5', '6', 'FX', 'Z', 'J']
NIEVE     = ['2026-02-23', '2026-02-24']

df_sl = df_te_s[~df_te_s['_route_id'].isin(LINEAS_AF)]
df_sn = df_te_s[~df_te_s['_date'].dt.strftime('%Y-%m-%d').isin(NIEVE)]
df_lp = df_sn[~df_sn['_route_id'].isin(LINEAS_AF)]

def mr(df): return mean_absolute_error(df[TARGET], df['_pred']), r2_score(df[TARGET], df['_pred'])

mg, rg   = mr(df_te_s)
msl, rsl = mr(df_sl)
msn, rsn = mr(df_sn)
mlp, rlp = mr(df_lp)

print('=== MAE real modelo sliding: impacto de exclusiones ===\n')
print(f'Global (100%):                    MAE={mg:.1f}s   R²={rg:.4f}')
print(f'Sin líneas anómalas ({len(df_sl)/len(df_te_s)*100:.1f}%):    MAE={msl:.1f}s   R²={rsl:.4f}')
print(f'Sin nevada ({len(df_sn)/len(df_te_s)*100:.1f}%):              MAE={msn:.1f}s   R²={rsn:.4f}')
print(f'Sin líneas + sin nevada ({len(df_lp)/len(df_te_s)*100:.1f}%): MAE={mlp:.1f}s   R²={rlp:.4f}')
print(f'\nReferencia val oct-dic 2025:      MAE≈130s')

## 7. Importancia de features vs drift

In [ ]:
feat_names = model_sw.feature_name()
imp_gain   = model_sw.feature_importance(importance_type='gain')
imp_gain_pct = 100 * imp_gain / imp_gain.sum()

rows = []
for feat, ig in zip(feat_names, imp_gain_pct):
    if feat not in df_tr_s.columns or feat not in df_te_s.columns:
        continue
    s_tr = df_tr_s[feat].dropna()
    s_te = df_te_s[feat].dropna()
    if len(s_tr) == 0 or len(s_te) == 0:
        continue
    mean_tr, mean_te = s_tr.mean(), s_te.mean()
    std_tr           = s_tr.std()
    drift = abs(mean_te - mean_tr) / (std_tr + 1e-9)
    rows.append({
        'feature':      feat,
        'imp_gain_%':   round(ig, 2),
        'mean_train':   round(mean_tr, 2),
        'mean_test':    round(mean_te, 2),
        'drift_sigmas': round(drift, 3),
        'score':        round(ig * drift, 3),
    })

imp_df = pd.DataFrame(rows).sort_values('imp_gain_%', ascending=False).reset_index(drop=True)
print('=== Importancia × drift (modelo sliding) ===')
print(imp_df.to_string(index=False))

## 8. Conclusiones

### Causa del empeoramiento

El modelo no tiene un fallo intrínseco. El problema es **concept drift**: en febrero 2026  
el retraso medio del metro subió un ~15% respecto a 2025. Las features más importantes  
del modelo (`delay_x_stops_remaining`, `delay_seconds`, `route_rolling_delay`) se  
desplazaron todas en la misma dirección, causando sobreestimación sistemática.

### Resultados

| Configuración | MAE (feb 2026) | R² | Sesgo línea B |
|---|---|---|---|
| Validación (oct-dic 2025) | ~130s | 0.57 | — |
| Train completo (ene-dic 2025 + ene 2026) | ~163s | 0.52 | +70s |
| **Sliding window (jul-dic 2025 + ene 2026)** | **~141s** | **0.57** | **+9s** |

### Lección para producción

La ventana deslizante recupera **22s de MAE** y elimina prácticamente el sesgo sistemático.  
Para un sistema en producción, el modelo debería re-entrenarse mensualmente con una  
ventana fija de los últimos 6-7 meses para adaptarse al drift de forma continua.